In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
!pip install torch_geometric -q

from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/DATN/data_new (1).csv")
df.head()

,ifInOctets11,ifOutOctets11,ifoutDiscards11,ifInUcastPkts11,ifInNUcastPkts11,ifInDiscards11,ifOutUcastPkts11,ifOutNUcastPkts11,tcpOutRsts,tcpInSegs,...,ipForwDatagrams,ipOutNoRoutes,ipInAddrErrors,icmpInMsgs,icmpInDestUnreachs,icmpOutMsgs,icmpOutDestUnreachs,icmpInEchos,icmpOutEchoReps,class
0,1720345568,950817778,0,50125711,18476,0,7768014,4370,1,688,...,50968341,8,0,55,30,42,21,22,23,normal
1,1758285451,883126781,0,48565201,15084,0,7603028,3751,1,635,...,65316058,7,0,50,27,52,23,24,25,normal
2,1942119779,1040643930,0,46257936,19024,0,6673231,4456,1,728,...,63625431,7,0,55,28,43,25,20,23,normal
3,2420591656,932582625,0,51592376,16347,0,7532024,4454,1,652,...,52214841,6,0,58,24,42,20,27,23,normal
4,2452598709,958032535,0,57723452,18827,0,6964137,4422,1,800,...,65600193,6,0,51,28,41,25,27,23,normal


In [ ]:
df_normal_all = df[df['class'] == 'normal'].sample(frac=1, random_state=42).reset_index(drop=True)
df_attacks = df[df['class'] != 'normal'].reset_index(drop=True)

df_normal_train = df_normal_all.iloc[:500].reset_index(drop=True)
df_normal_test  = df_normal_all.iloc[500:600].reset_index(drop=True)

print("Train normal:", len(df_normal_train))
print("Test normal :", len(df_normal_test))
print("Attack types:", df_attacks['class'].unique())

Train normal: 500
Test normal : 100
Attack types: ['icmp-echo' 'tcp-syn' 'udp-flood' 'httpFlood' 'slowloris' 'slowpost'
 'bruteForce']


In [ ]:
groups = {
    'Interface': ['ifInOctets11','ifOutOctets11','ifoutDiscards11','ifInUcastPkts11',
                  'ifInNUcastPkts11','ifInDiscards11','ifOutUcastPkts11','ifOutNUcastPkts11'],
    'IP': ['ipInReceives','ipInDelivers','ipOutRequests','ipOutDiscards',
           'ipInDiscards','ipForwDatagrams','ipOutNoRoutes','ipInAddrErrors'],
    'TCP': ['tcpOutRsts','tcpInSegs','tcpOutSegs','tcpPassiveOpens',
            'tcpRetransSegs','tcpCurrEstab','tcpEstabResets','tcp?ActiveOpens'],
    'UDP': ['udpInDatagrams','udpOutDatagrams','udpInErrors','udpNoPorts'],
    'ICMP': ['icmpInMsgs','icmpInDestUnreachs','icmpOutMsgs',
             'icmpOutDestUnreachs','icmpInEchos','icmpOutEchoReps']
}

if 'tcp?ActiveOpens' in df.columns:
    df['tcpActiveOpens'] = df['tcp?ActiveOpens']

group_cols = {
    'Interface': groups['Interface'],
    'IP': groups['IP'],
    'TCP': [c if c != 'tcp?ActiveOpens' else 'tcpActiveOpens' for c in groups['TCP']],
    'UDP': groups['UDP'],
    'ICMP': groups['ICMP']
}

node_order = ['Interface','IP','TCP','UDP','ICMP']
max_dim = max(len(cols) for cols in group_cols.values())

print("Max feature dim:", max_dim)


Max feature dim: 8


In [ ]:
def create_features(df_set, scalers=None, fit=True):
    N = len(df_set)
    x = torch.zeros(N, 5, max_dim)

    if fit:
        scalers = {}

    for i, group in enumerate(node_order):
        cols = [c for c in group_cols[group] if c in df_set.columns]
        values = df_set[cols].values.astype(float)

        if fit:
            scaler = StandardScaler()
            values = scaler.fit_transform(values)
            scalers[group] = scaler
        else:
            scaler = scalers[group]
            values = scaler.transform(values)

        values = torch.tensor(values, dtype=torch.float)
        x[:, i, :values.shape[1]] = values

    return x, scalers


In [ ]:
x_train, scalers = create_features(df_normal_train, fit=True)

# time embedding
N_train = len(x_train)
time_emb = torch.sin(torch.linspace(0, 2*np.pi, N_train)).view(N_train,1,1)
time_emb = time_emb.repeat(1,5,1)

x_train = torch.cat([x_train, time_emb], dim=-1)

in_channels = x_train.shape[-1]

edge_index = torch.tensor([
    [0,1],[1,0],
    [1,2],[2,1],
    [1,3],[3,1],
    [1,4],[4,1]
]).t().long()

data_list_train = [
    Data(x=x_train[i], edge_index=edge_index.clone())
    for i in range(N_train)
]

print("Train graphs:", len(data_list_train))

Train graphs: 500


In [ ]:
x_test_normal, _ = create_features(df_normal_test, scalers=scalers, fit=False)

N_test_normal = len(x_test_normal)
time_emb = torch.sin(torch.linspace(0, 2*np.pi, N_test_normal)).view(N_test_normal,1,1)
time_emb = time_emb.repeat(1,5,1)

x_test_normal = torch.cat([x_test_normal, time_emb], dim=-1)

data_list_normal_test = [
    Data(x=x_test_normal[i], edge_index=edge_index.clone())
    for i in range(N_test_normal)
]

print("Test normal graphs:", len(data_list_normal_test))

Test normal graphs: 100


In [ ]:
data_list_attacks = {}

for attack in df_attacks['class'].unique():
    df_attack_type = df_attacks[df_attacks['class']==attack].reset_index(drop=True)

    x_attack, _ = create_features(df_attack_type, scalers=scalers, fit=False)

    N_attack = len(x_attack)
    time_emb = torch.sin(torch.linspace(0,2*np.pi,N_attack)).view(N_attack,1,1)
    time_emb = time_emb.repeat(1,5,1)

    x_attack = torch.cat([x_attack, time_emb], dim=-1)

    data_list_attacks[attack] = [
        Data(x=x_attack[i], edge_index=edge_index.clone())
        for i in range(N_attack)
    ]

print("Attack types prepared:", list(data_list_attacks.keys()))

Attack types prepared: ['icmp-echo', 'tcp-syn', 'udp-flood', 'httpFlood', 'slowloris', 'slowpost', 'bruteForce']


In [ ]:
class GAT_AE(nn.Module):
    def __init__(self, in_channels, hidden=256, out=128):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden, heads=8, concat=True, dropout=0.2)
        self.gat2 = GATConv(hidden*8, hidden, heads=8, concat=True, dropout=0.2)
        self.gat3 = GATConv(hidden*8, out, heads=1, concat=False)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        emb = self.gat3(x, edge_index)

        adj_logits = torch.matmul(emb, emb.t())
        return adj_logits

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GAT_AE(in_channels).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, patience=10)
criterion = nn.BCEWithLogitsLoss()

adj_target = torch.zeros(5,5).to(device)
adj_target[0,1]=adj_target[1,0]=1.2
adj_target[1,2]=adj_target[2,1]=2
adj_target[1,3]=adj_target[3,1]=1.2
adj_target[1,4]=adj_target[4,1]=1.2

loader = DataLoader(data_list_train, batch_size=64, shuffle=True)

best_loss = 1e9
patience = 40
counter = 0

for epoch in range(300):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        adj_logits = model(batch)

        batch_size = batch.num_graphs
        adj_batch = torch.block_diag(*([adj_target]*batch_size))

        loss = criterion(adj_logits, adj_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(loader)
    scheduler.step(avg_loss)

    if epoch%10==0:
        print(epoch, avg_loss)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "/content/best_vanilla_gat.pth")
        counter=0
    else:
        counter+=1
        if counter>=patience:
            print("Early stopping")
            break

0 1.0768638029694557
10 0.6957013159990311
20 0.6947543099522591
30 0.6933463290333748
40 0.6933701783418655
50 0.6932692900300026
60 0.6931570619344711
70 0.6932706460356712
80 0.6932415664196014
90 0.6933577731251717
100 0.6932226270437241
Early stopping


In [ ]:
model.load_state_dict(torch.load("/content/best_vanilla_gat.pth"))
model.eval()

train_scores = []

with torch.no_grad():
    for data in data_list_train:
        data = data.to(device)
        logits = model(data)
        score = F.binary_cross_entropy_with_logits(logits, adj_target).item()
        train_scores.append(score)

threshold = np.mean(train_scores) + 2*np.std(train_scores)

print("Threshold:", threshold)

Threshold: 0.6966900790251785


In [ ]:
false_positive = 0

with torch.no_grad():
    for data in data_list_normal_test:
        data = data.to(device)
        logits = model(data)
        score = F.binary_cross_entropy_with_logits(logits, adj_target).item()

        if score > threshold:
            false_positive += 1

FPR = false_positive / len(data_list_normal_test)

print("FPR:", FPR)

FPR: 0.05


In [ ]:
for attack, data_list in data_list_attacks.items():
    detected = 0

    with torch.no_grad():
        for data in data_list:
            data = data.to(device)
            logits = model(data)
            score = F.binary_cross_entropy_with_logits(logits, adj_target).item()

            if score > threshold:
                detected += 1

    recall = detected / len(data_list)

    print(f"\n{attack}")
    print("Total :", len(data_list))
    print("Detected :", detected)
    print("Recall :", recall)


icmp-echo
Total : 632
Detected : 569
Recall : 0.9003164556962026

tcp-syn
Total : 960
Detected : 960
Recall : 1.0

udp-flood
Total : 773
Detected : 765
Recall : 0.9896507115135834

httpFlood
Total : 573
Detected : 573
Recall : 1.0

slowloris
Total : 780
Detected : 611
Recall : 0.7833333333333333

slowpost
Total : 480
Detected : 480
Recall : 1.0

bruteForce
Total : 200
Detected : 200
Recall : 1.0
